# Amazon Top-K · TOP 1 / TOP 2 / TOP 3 최종 제출 파일 생성

**목표:** 이전 `UltraGCN + SimpleX Medium Ablation`에서 검증 성능 1~3위였던 앙상블을 **전체 공식 `train.csv`로 처음부터 재학습하여** 세 개의 독립적인 CSV로 생성.

| 제출 파일 | Validation Recall@20 | 구성 | 결합 |
|---|---:|---|---|
| `submission_top1.csv` | 0.042812 | LightGCN 35%, SimpleX 50%, UltraGCN 15% | RRF k=60 |
| `submission_top2.csv` | 0.042768 | LightGCN 35%, SimpleX 50%, UltraGCN 15% | RRF k=20 |
| `submission_top3.csv` | 0.041601 | LightGCN 50%, SimpleX 50% | RRF k=20 |

**재학습은 총 3회만:** LightGCN 575 Epoch, SimpleX X01_gamma05_700의 best_epoch **650**, UltraGCN U04_lr001의 best_epoch **80**. 세 앙상블에는 추가 모델 학습이 필요하지 않습니다.

**중요:** 위 Recall 값은 *Val* 평가 점수이고, 새로 생성된 전체 학습 모델의 Kaggle Test 점수는 제출해야 알 수 있습니다. 랭킹에 반복적 하이퍼파라미터 탐색을 했으므로 Validation 과적합 가능성이 있습니다. 원본 논문을 완전히 재현한 UltraGCN / SimpleX가 아니라 앞선 실험에서 사용한 경량 구현을 **변경 없이 재사용**합니다.

**입력:** 대회 공식 `train.csv` 및 `test.csv`를 Kaggle `Add Input`으로 연결하세요. 공통 Validation ZIP은 이 노트북에서 필요하지 않습니다. **GPU를 켜세요.**

## 1. 환경 확인 및 데이터 경로

In [ ]:
from pathlib import Path
from collections import defaultdict
import csv, hashlib, json, os, subprocess, sys, zipfile
import pandas as pd
import torch
from IPython.display import display

WORK=Path('/kaggle/working')
INPUT=Path('/kaggle/input')
OUT=WORK/'top3_final'
OUT.mkdir(parents=True, exist_ok=True)
SEED=42
DEVICE='cuda:0'

if not torch.cuda.is_available():
    raise RuntimeError('GPU가 꺼져 있습니다. Kaggle Settings → Accelerator → GPU T4를 선택하세요.')
roots=sorted({p.parent for p in INPUT.rglob('train.csv') if (p.parent/'test.csv').is_file()})
if len(roots)!=1:
    raise RuntimeError(f'공식 train.csv/test.csv 파일 쌍을 한 곳에서 찾지 못했습니다. 검색 결과={roots}')
DATA=roots[0]
TRAIN=DATA/'train.csv'
TEST=DATA/'test.csv'
print('GPU:',torch.cuda.get_device_name(0))
print('원본 공식 학습 데이터:',TRAIN)
print('Test 사용자 파일:',TEST)
print('최종 출력 위치:',OUT)

## 2. 이전 실험과 동일한 예측 형식 검사 코드

In [ ]:
%%writefile /kaggle/working/teamkit.py
"""Shared, standard-library-only evaluation utilities for the Amazon Top-20 team.

All interactions are targets, regardless of rating. IDs are kept as strings so
CSV round-trips do not change them. The public CSV contract is user_id,item_id.
"""

from __future__ import annotations

import argparse
import csv
import hashlib
import io
import json
import math
import os
import sys
import tempfile
import zipfile
from collections import Counter, defaultdict
from contextlib import contextmanager
from pathlib import Path
from typing import Iterable, Mapping, Sequence


TOP_K = 20
SPLIT_SEED = 42
HOLDOUT_FRACTION = 0.20


@contextmanager
def open_source(path: str | None, competition_zip: str | None, member: str):
    """Open a UTF-8 file, either directly or as a named competition ZIP member."""
    if (path is None) == (competition_zip is None):
        raise ValueError("Provide exactly one of a file path or --competition-zip")
    if competition_zip is not None:
        with zipfile.ZipFile(competition_zip) as archive:
            with archive.open(member) as raw:
                with io.TextIOWrapper(raw, encoding="utf-8-sig", newline="") as stream:
                    yield stream
    else:
        with open(path, "r", encoding="utf-8-sig", newline="") as stream:
            yield stream


def _reader(stream, required: Sequence[str]) -> csv.DictReader:
    reader = csv.DictReader(stream)
    if reader.fieldnames is None or not set(required).issubset(reader.fieldnames):
        raise ValueError(f"Required columns: {', '.join(required)}")
    return reader


def _clean_id(value: str | None, name: str) -> str:
    if value is None or value == "" or value != value.strip() or any(c.isspace() for c in value):
        raise ValueError(f"Invalid {name}: {value!r}")
    return value


def _stable_rank(seed: int, user_id: str, item_id: str) -> bytes:
    payload = f"{seed}\0{user_id}\0{item_id}".encode("utf-8")
    return hashlib.blake2b(payload, digest_size=16).digest()


def _sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _source_sha256(path: str | None, competition_zip: str | None, member: str) -> str:
    digest = hashlib.sha256()
    if competition_zip is not None:
        with zipfile.ZipFile(competition_zip) as archive, archive.open(member) as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(chunk)
    else:
        with open(path, "rb") as stream:
            for chunk in iter(lambda: stream.read(1024 * 1024), b""):
                digest.update(chunk)
    return digest.hexdigest()


def make_split(
    train_path: str | None,
    competition_zip: str | None,
    output_dir: str | Path,
    seed: int = SPLIT_SEED,
    fraction: float = HOLDOUT_FRACTION,
    overwrite: bool = False,
) -> dict:
    """Make a stable, per-user split on unique user-item pairs.

    Hold out ceil(fraction * interactions), at most 20, leaving at least one
    interaction in fit. The hash rank is independent of input row order.
    """
    if not 0 < fraction < 1:
        raise ValueError("fraction must be between 0 and 1")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    names = ("val_train.csv", "val_truth.csv", "val_users.csv", "manifest.json")
    existing = [name for name in names if (output_dir / name).exists()]
    if existing and not overwrite:
        raise FileExistsError(f"Existing split files: {existing}; use --overwrite to replace")

    with open_source(train_path, competition_zip, "train.csv") as stream:
        reader = _reader(stream, ("user_id", "item_id"))
        columns = list(reader.fieldnames or [])
        rows: list[tuple[str, ...]] = []
        user_items: dict[str, list[str]] = defaultdict(list)
        pairs: set[tuple[str, str]] = set()
        for row in reader:
            if None in row:
                raise ValueError("Malformed train.csv row with extra fields")
            user_id = _clean_id(row["user_id"], "user_id")
            item_id = _clean_id(row["item_id"], "item_id")
            pair = (user_id, item_id)
            if pair in pairs:
                raise ValueError(f"Duplicate interaction: {pair}")
            pairs.add(pair)
            user_items[user_id].append(item_id)
            rows.append(tuple(row[column] for column in columns))

    held: set[tuple[str, str]] = set()
    for user_id, items in user_items.items():
        if len(items) < 2:
            continue
        n_held = min(TOP_K, len(items) - 1, max(1, math.ceil(fraction * len(items))))
        chosen = sorted(items, key=lambda item: (_stable_rank(seed, user_id, item), item))[:n_held]
        held.update((user_id, item) for item in chosen)

    fit_path = output_dir / "val_train.csv"
    truth_path = output_dir / "val_truth.csv"
    users_path = output_dir / "val_users.csv"
    user_col, item_col = columns.index("user_id"), columns.index("item_id")
    with fit_path.open("w", encoding="utf-8", newline="") as fit_stream, truth_path.open(
        "w", encoding="utf-8", newline=""
    ) as truth_stream:
        fit_writer, truth_writer = csv.writer(fit_stream), csv.writer(truth_stream)
        fit_writer.writerow(columns)
        truth_writer.writerow(("user_id", "item_id"))
        for row in rows:
            pair = (row[user_col], row[item_col])
            if pair in held:
                truth_writer.writerow(pair)
            else:
                fit_writer.writerow(row)

    with users_path.open("w", encoding="utf-8", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(("user_id",))
        for user_id in user_items:
            if len(user_items[user_id]) >= 2:
                writer.writerow((user_id,))

    source_hash = _source_sha256(train_path, competition_zip, "train.csv")
    split_key = f"{source_hash}|blake2b16|{seed}|{fraction}|ceil20keep1"
    manifest = {
        "split_id": hashlib.sha256(split_key.encode()).hexdigest()[:16],
        "algorithm": "per-user blake2b-128 rank over seed, user_id, item_id",
        "seed": seed,
        "holdout_fraction": fraction,
        "holdout_count": "min(20, n-1, max(1, ceil(fraction*n))) for n>=2",
        "source_train_sha256": source_hash,
        "source_rows": len(rows),
        "source_users": len(user_items),
        "fit_rows": len(rows) - len(held),
        "truth_rows": len(held),
        "validation_users": sum(len(items) >= 2 for items in user_items.values()),
        "files_sha256": {name: _sha256(output_dir / name) for name in names[:3]},
    }
    with (output_dir / "manifest.json").open("w", encoding="utf-8") as stream:
        json.dump(manifest, stream, ensure_ascii=False, indent=2)
        stream.write("\n")
    return manifest


def verify_split(output_dir: str | Path) -> dict:
    output_dir = Path(output_dir)
    with (output_dir / "manifest.json").open("r", encoding="utf-8") as stream:
        manifest = json.load(stream)
    for name, expected in manifest["files_sha256"].items():
        actual = _sha256(output_dir / name)
        if actual != expected:
            raise ValueError(f"Split file checksum mismatch: {name}")
    return manifest


def read_users(path: str | Path) -> list[str]:
    users: list[str] = []
    seen: set[str] = set()
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        for row in _reader(stream, ("user_id",)):
            if None in row:
                raise ValueError(f"Malformed user CSV row in {path}")
            user_id = _clean_id(row["user_id"], "user_id")
            if user_id in seen:
                raise ValueError(f"Duplicate user in {path}: {user_id}")
            seen.add(user_id)
            users.append(user_id)
    return users


def read_interactions(path: str | Path) -> tuple[dict[str, set[str]], Counter[str]]:
    """Return seen items by user and item popularity from a fit CSV."""
    seen: dict[str, set[str]] = defaultdict(set)
    popularity: Counter[str] = Counter()
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        for row in _reader(stream, ("user_id", "item_id")):
            if None in row:
                raise ValueError(f"Malformed interaction CSV row in {path}")
            user_id = _clean_id(row["user_id"], "user_id")
            item_id = _clean_id(row["item_id"], "item_id")
            if item_id not in seen[user_id]:
                seen[user_id].add(item_id)
                popularity[item_id] += 1
    return dict(seen), popularity


def read_truth(path: str | Path) -> dict[str, set[str]]:
    truth: dict[str, set[str]] = defaultdict(set)
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        for row in _reader(stream, ("user_id", "item_id")):
            if None in row:
                raise ValueError(f"Malformed truth CSV row in {path}")
            user_id = _clean_id(row["user_id"], "user_id")
            item_id = _clean_id(row["item_id"], "item_id")
            if item_id in truth[user_id]:
                raise ValueError(f"Duplicate truth item for user {user_id}: {item_id}")
            truth[user_id].add(item_id)
    return dict(truth)


def read_predictions(path: str | Path) -> dict[str, list[str]]:
    predictions: dict[str, list[str]] = {}
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames != ["user_id", "item_id"]:
            raise ValueError("Prediction columns must be exactly user_id,item_id")
        for row in reader:
            if None in row:
                raise ValueError("Malformed prediction CSV row")
            user_id = _clean_id(row["user_id"], "user_id")
            if user_id in predictions:
                raise ValueError(f"Duplicate prediction user: {user_id}")
            value = row["item_id"]
            if value is None:
                raise ValueError(f"Missing item list for user {user_id}")
            items = value.split(" ")
            if len(items) != TOP_K or any(not item or item != item.strip() for item in items):
                raise ValueError(f"User {user_id} needs exactly {TOP_K} space-separated item IDs")
            if len(set(items)) != TOP_K:
                raise ValueError(f"Duplicate recommended item for user {user_id}")
            predictions[user_id] = items
    return predictions


def validate_predictions(
    predictions: Mapping[str, Sequence[str]],
    expected_users: Iterable[str],
    seen: Mapping[str, set[str]],
    catalog: set[str],
) -> None:
    expected = set(expected_users)
    actual = set(predictions)
    if actual != expected:
        missing, extra = expected - actual, actual - expected
        raise ValueError(f"User mismatch: {len(missing)} missing, {len(extra)} extra")
    for user_id, items in predictions.items():
        if len(items) != TOP_K or len(set(items)) != TOP_K:
            raise ValueError(f"User {user_id} needs {TOP_K} distinct items")
        unknown = set(items) - catalog
        if unknown:
            raise ValueError(f"Unknown item for user {user_id}: {sorted(unknown)[:3]}")
        repeated = set(items) & seen.get(user_id, set())
        if repeated:
            raise ValueError(f"Seen item recommended to user {user_id}: {sorted(repeated)[:3]}")


def recall_at_20(predictions: Mapping[str, Sequence[str]], truth: Mapping[str, set[str]]) -> dict:
    """Competition-style pooled Recall@20, not mean of user-level recalls."""
    if set(predictions) != set(truth):
        raise ValueError("Prediction and truth user sets differ")
    hits = sum(len(set(items[:TOP_K]) & truth[user_id]) for user_id, items in predictions.items())
    denominator = sum(min(TOP_K, len(items)) for items in truth.values())
    if denominator == 0:
        raise ValueError("No validation truth items")
    return {"recall_at_20": hits / denominator, "hits": hits, "denominator": denominator, "users": len(truth)}


def top20_unseen(
    user_id: str,
    ranked_candidates: Iterable[str],
    seen: Mapping[str, set[str]],
    fallback_items: Iterable[str],
    catalog: set[str],
) -> list[str]:
    """Filter a model ranking; fill short lists with popular unseen items."""
    chosen: list[str] = []
    used = set(seen.get(user_id, set()))
    for source in (ranked_candidates, fallback_items):
        for item_id in source:
            item_id = str(item_id)
            if item_id not in catalog or item_id in used:
                continue
            chosen.append(item_id)
            used.add(item_id)
            if len(chosen) == TOP_K:
                return chosen
    raise ValueError(f"Fewer than {TOP_K} available unseen items for user {user_id}")


def write_predictions(path: str | Path, users: Iterable[str], predictions: Mapping[str, Sequence[str]]) -> None:
    """Write one row per user in the provided order, using single spaces."""
    with open(path, "w", encoding="utf-8", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(("user_id", "item_id"))
        for user_id in users:
            items = predictions[user_id]
            if len(items) != TOP_K or len(set(items)) != TOP_K:
                raise ValueError(f"User {user_id} needs {TOP_K} distinct items")
            writer.writerow((user_id, " ".join(map(str, items))))


def filter_reviews(
    fit_path: str | Path,
    output_path: str | Path,
    reviews_path: str | None = None,
    competition_zip: str | None = None,
) -> dict:
    """Keep only training-pair reviews to prevent validation target leakage."""
    seen, _ = read_interactions(fit_path)
    allowed = {(user_id, item_id) for user_id, items in seen.items() for item_id in items}
    found: set[tuple[str, str]] = set()
    written = 0
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    descriptor, temporary_path = tempfile.mkstemp(prefix=".reviews-", suffix=".tmp", dir=output_path.parent)
    try:
        with os.fdopen(descriptor, "w", encoding="utf-8", newline="") as output, open_source(
            reviews_path, competition_zip, "train_reviews.jsonl"
        ) as source:
            for line in source:
                review = json.loads(line)
                pair = (str(review["user_id"]), str(review["item_id"]))
                if pair in allowed:
                    output.write(line)
                    found.add(pair)
                    written += 1
        if found != allowed:
            raise ValueError(f"Review coverage mismatch: {len(allowed - found)} fit pairs have no review")
        os.replace(temporary_path, output_path)
    finally:
        if os.path.exists(temporary_path):
            os.unlink(temporary_path)
    return {"written_reviews": written, "distinct_fit_pairs": len(allowed)}


def _print_json(value: dict) -> None:
    print(json.dumps(value, ensure_ascii=False, indent=2))


def main(argv: Sequence[str] | None = None) -> int:
    parser = argparse.ArgumentParser(description="Shared Amazon Top-20 evaluation tools")
    commands = parser.add_subparsers(dest="command", required=True)

    split = commands.add_parser("split", help="Create deterministic validation CSVs")
    source = split.add_mutually_exclusive_group(required=True)
    source.add_argument("--train", help="Path to extracted train.csv")
    source.add_argument("--competition-zip", help="Path to original competition ZIP")
    split.add_argument("--out", required=True, help="Output directory")
    split.add_argument("--seed", type=int, default=SPLIT_SEED)
    split.add_argument("--fraction", type=float, default=HOLDOUT_FRACTION)
    split.add_argument("--overwrite", action="store_true")

    verify = commands.add_parser("verify-split", help="Check shared split file hashes")
    verify.add_argument("--dir", required=True)

    score = commands.add_parser("score", help="Validate and score validation predictions")
    score.add_argument("--pred", required=True)
    score.add_argument("--fit", required=True)
    score.add_argument("--truth", required=True)

    check = commands.add_parser("check", help="Validate a final test submission")
    check.add_argument("--pred", required=True)
    check.add_argument("--train", required=True)
    check.add_argument("--test", required=True)

    reviews = commands.add_parser("filter-reviews", help="Remove held-out reviews")
    reviews.add_argument("--fit", required=True)
    review_source = reviews.add_mutually_exclusive_group(required=True)
    review_source.add_argument("--reviews", help="Path to extracted train_reviews.jsonl")
    review_source.add_argument("--competition-zip", help="Path to original competition ZIP")
    reviews.add_argument("--out", required=True)

    args = parser.parse_args(argv)
    if args.command == "split":
        _print_json(make_split(args.train, args.competition_zip, args.out, args.seed, args.fraction, args.overwrite))
    elif args.command == "verify-split":
        manifest = verify_split(args.dir)
        _print_json({"split_id": manifest["split_id"], "verified": True})
    elif args.command == "score":
        seen, popularity = read_interactions(args.fit)
        truth = read_truth(args.truth)
        predictions = read_predictions(args.pred)
        catalog = set(popularity) | {item for items in truth.values() for item in items}
        validate_predictions(predictions, truth, seen, catalog)
        result = recall_at_20(predictions, truth)
        result["valid_submission_format"] = True
        _print_json(result)
    elif args.command == "check":
        seen, popularity = read_interactions(args.train)
        users = read_users(args.test)
        predictions = read_predictions(args.pred)
        validate_predictions(predictions, users, seen, set(popularity))
        _print_json({"valid": True, "users": len(users), "items_per_user": TOP_K})
    elif args.command == "filter-reviews":
        _print_json(filter_reviews(args.fit, args.out, args.reviews, args.competition_zip))
    return 0


if __name__ == "__main__":
    try:
        sys.exit(main())
    except (ValueError, FileExistsError, FileNotFoundError, KeyError, zipfile.BadZipFile) as error:
        print(f"ERROR: {error}", file=sys.stderr)
        sys.exit(1)

## 3. 이전 실험과 동일한 LightGCN 실행 코드

In [ ]:
%%writefile /kaggle/working/graph_models.py
"""Single-GPU LightGCN / NGCF experiments for the shared Amazon Top-20 split.

Example:
    python graph_models.py --model lightgcn \
      --fit ../amazon_topk_teamkit/validation/val_train.csv \
      --users ../amazon_topk_teamkit/validation/val_users.csv \
      --truth ../amazon_topk_teamkit/validation/val_truth.csv \
      --out lightgcn_val.csv

No metadata or reviews are read. Train interactions are the only graph edges.
"""

from __future__ import annotations

import argparse
import copy
import csv
import json
import math
import random
import time
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Sequence

try:
    import torch
    from torch import nn
    from torch.nn import functional as F
except ModuleNotFoundError:
    torch = None
    nn = None
    F = None


TOP_K = 20
_BaseModel = nn.Module if nn is not None else object


def _id_key(value: str):
    return (0, int(value)) if value.isdecimal() else (1, value)


def _csv_rows(path: str | Path, required: tuple[str, ...]):
    with open(path, "r", encoding="utf-8-sig", newline="") as stream:
        reader = csv.DictReader(stream)
        if reader.fieldnames is None or not set(required).issubset(reader.fieldnames):
            raise ValueError(f"{path} must contain {', '.join(required)}")
        for row in reader:
            if None in row:
                raise ValueError(f"Malformed CSV row in {path}")
            yield row


def _identifier(value: str | None, column: str) -> str:
    if not value or value != value.strip() or any(char.isspace() for char in value):
        raise ValueError(f"Invalid {column}: {value!r}")
    return value


@dataclass
class InteractionData:
    user_ids: list[str]
    item_ids: list[str]
    user_index: dict[str, int]
    item_index: dict[str, int]
    positive_users: list[int]
    positive_items: list[int]
    seen_items: dict[int, set[int]]
    target_users: list[str]

    @property
    def num_users(self) -> int:
        return len(self.user_ids)

    @property
    def num_items(self) -> int:
        return len(self.item_ids)

    @property
    def num_edges(self) -> int:
        return len(self.positive_users)


def load_data(fit_path: str | Path, users_path: str | Path) -> InteractionData:
    """Index unique positive edges; do not filter interactions by rating."""
    edges: set[tuple[str, str]] = set()
    for row in _csv_rows(fit_path, ("user_id", "item_id")):
        edges.add((_identifier(row["user_id"], "user_id"), _identifier(row["item_id"], "item_id")))
    if not edges:
        raise ValueError("Empty fit interactions")

    target_users: list[str] = []
    target_set: set[str] = set()
    for row in _csv_rows(users_path, ("user_id",)):
        user_id = _identifier(row["user_id"], "user_id")
        if user_id in target_set:
            raise ValueError(f"Duplicate target user: {user_id}")
        target_set.add(user_id)
        target_users.append(user_id)

    train_users = {user for user, _ in edges}
    unseen_users = target_set - train_users
    if unseen_users:
        raise ValueError(f"{len(unseen_users)} target users have no training interaction")
    user_ids = sorted(train_users, key=_id_key)
    item_ids = sorted({item for _, item in edges}, key=_id_key)
    if len(item_ids) <= TOP_K:
        raise ValueError("Need more than 20 known items to recommend 20 unseen items")
    user_index = {value: index for index, value in enumerate(user_ids)}
    item_index = {value: index for index, value in enumerate(item_ids)}
    positive_users: list[int] = []
    positive_items: list[int] = []
    seen_items: dict[int, set[int]] = defaultdict(set)
    for user, item in sorted(edges, key=lambda pair: (_id_key(pair[0]), _id_key(pair[1]))):
        u, i = user_index[user], item_index[item]
        positive_users.append(u)
        positive_items.append(i)
        seen_items[u].add(i)
    for user in target_users:
        if len(seen_items[user_index[user]]) + TOP_K > len(item_ids):
            raise ValueError(f"User {user} has fewer than 20 unseen items")
    return InteractionData(
        user_ids, item_ids, user_index, item_index, positive_users,
        positive_items, dict(seen_items), target_users,
    )


def load_truth(path: str | Path, target_users: Sequence[str]) -> dict[str, set[str]]:
    truth: dict[str, set[str]] = defaultdict(set)
    for row in _csv_rows(path, ("user_id", "item_id")):
        user, item = _identifier(row["user_id"], "user_id"), _identifier(row["item_id"], "item_id")
        if item in truth[user]:
            raise ValueError(f"Duplicate truth pair: {user}, {item}")
        truth[user].add(item)
    if set(truth) != set(target_users):
        raise ValueError("Truth and target user sets differ")
    return dict(truth)


def recall_at_20(predictions: dict[str, list[str]], truth: dict[str, set[str]]) -> dict:
    """Pooled competition Recall@20: total hits / sum(min(20, |GT_u|))."""
    if set(predictions) != set(truth):
        raise ValueError("Prediction and truth user sets differ")
    hits = sum(len(set(items[:TOP_K]) & truth[user]) for user, items in predictions.items())
    denominator = sum(min(TOP_K, len(items)) for items in truth.values())
    if not denominator:
        raise ValueError("No truth items")
    return {"recall_at_20": hits / denominator, "hits": hits, "denominator": denominator, "users": len(truth)}


def normalized_graph(data: InteractionData, device):
    """D^-1/2 A D^-1/2 for the undirected user-item graph."""
    if torch is None:
        raise RuntimeError("PyTorch is required to build the graph")
    u = torch.tensor(data.positive_users, dtype=torch.long)
    i = torch.tensor(data.positive_items, dtype=torch.long) + data.num_users
    rows = torch.cat((u, i))
    cols = torch.cat((i, u))
    degree = torch.bincount(rows, minlength=data.num_users + data.num_items).float()
    values = (degree[rows] * degree[cols]).rsqrt()
    indices = torch.stack((rows, cols)).to(device)
    return torch.sparse_coo_tensor(
        indices, values.to(device),
        (data.num_users + data.num_items, data.num_users + data.num_items),
        device=device,
    ).coalesce()


def sparse_edge_dropout(graph, probability: float):
    if probability <= 0:
        return graph
    if not 0 <= probability < 1:
        raise ValueError("Edge dropout must be in [0, 1)")
    keep = torch.rand(graph._nnz(), device=graph.device) >= probability
    return torch.sparse_coo_tensor(
        graph.indices()[:, keep], graph.values()[keep] / (1 - probability),
        graph.shape, device=graph.device,
    ).coalesce()


class LightGCN(_BaseModel):
    def __init__(self, n_users: int, n_items: int, dim: int, layers: int):
        if nn is None:
            raise RuntimeError("PyTorch is required to train LightGCN")
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, dim)
        self.item_embedding = nn.Embedding(n_items, dim)
        self.layers = layers
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)

    def forward(self, graph):
        current = torch.cat((self.user_embedding.weight, self.item_embedding.weight), dim=0)
        layer_outputs = [current]
        for _ in range(self.layers):
            current = torch.sparse.mm(graph, current)
            layer_outputs.append(current)
        final = torch.stack(layer_outputs, dim=0).mean(dim=0)
        return final[: self.user_embedding.num_embeddings], final[self.user_embedding.num_embeddings :]


class NGCF(_BaseModel):
    def __init__(self, n_users: int, n_items: int, dim: int, layers: int, message_dropout: float):
        if nn is None:
            raise RuntimeError("PyTorch is required to train NGCF")
        super().__init__()
        self.user_embedding = nn.Embedding(n_users, dim)
        self.item_embedding = nn.Embedding(n_items, dim)
        self.sum_linears = nn.ModuleList(nn.Linear(dim, dim) for _ in range(layers))
        self.bi_linears = nn.ModuleList(nn.Linear(dim, dim) for _ in range(layers))
        self.message_dropout = message_dropout
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)
        for layer in (*self.sum_linears, *self.bi_linears):
            nn.init.xavier_uniform_(layer.weight)
            nn.init.zeros_(layer.bias)

    def forward(self, graph):
        current = torch.cat((self.user_embedding.weight, self.item_embedding.weight), dim=0)
        layer_outputs = [current]
        for sum_linear, bi_linear in zip(self.sum_linears, self.bi_linears):
            neighbors = torch.sparse.mm(graph, current)
            current = F.leaky_relu(sum_linear(current + neighbors) + bi_linear(current * neighbors), negative_slope=0.2)
            current = F.dropout(current, p=self.message_dropout, training=self.training)
            current = F.normalize(current, p=2, dim=1)
            layer_outputs.append(current)
        final = torch.cat(layer_outputs, dim=1)
        return final[: self.user_embedding.num_embeddings], final[self.user_embedding.num_embeddings :]


def sample_triples(data: InteractionData, count: int, rng: random.Random, device):
    """Sample one uniformly random unseen item per sampled positive edge."""
    if count <= 0:
        raise ValueError("samples-per-epoch must be positive")
    edge_indices = [rng.randrange(data.num_edges) for _ in range(count)]
    users = [data.positive_users[index] for index in edge_indices]
    positive = [data.positive_items[index] for index in edge_indices]
    negative = [rng.randrange(data.num_items) for _ in range(count)]
    for position, (user, item) in enumerate(zip(users, negative)):
        if item in data.seen_items[user]:
            while True:
                candidate = rng.randrange(data.num_items)
                if candidate not in data.seen_items[user]:
                    negative[position] = candidate
                    break
    return (
        torch.tensor(users, dtype=torch.long, device=device),
        torch.tensor(positive, dtype=torch.long, device=device),
        torch.tensor(negative, dtype=torch.long, device=device),
    )


@torch.no_grad() if torch is not None else (lambda function: function)
def predict_top20(model, graph, data: InteractionData, user_batch_size: int) -> dict[str, list[str]]:
    """Score all known items in bounded user batches and mask fit interactions."""
    if user_batch_size <= 0:
        raise ValueError("eval-user-batch-size must be positive")
    model.eval()
    user_vectors, item_vectors = model(graph)
    item_matrix = item_vectors.T.contiguous()
    predictions: dict[str, list[str]] = {}
    for start in range(0, len(data.target_users), user_batch_size):
        batch = data.target_users[start : start + user_batch_size]
        user_indices = torch.tensor([data.user_index[user] for user in batch], dtype=torch.long, device=graph.device)
        scores = user_vectors[user_indices] @ item_matrix
        masked_rows: list[int] = []
        masked_columns: list[int] = []
        for local_row, user_index in enumerate(user_indices.tolist()):
            for item_index in data.seen_items[user_index]:
                masked_rows.append(local_row)
                masked_columns.append(item_index)
        if masked_rows:
            row_index = torch.tensor(masked_rows, dtype=torch.long, device=graph.device)
            column_index = torch.tensor(masked_columns, dtype=torch.long, device=graph.device)
            scores[row_index, column_index] = -float("inf")
        top = torch.topk(scores, k=TOP_K, dim=1).indices.cpu().tolist()
        for user_id, item_indices in zip(batch, top):
            predictions[user_id] = [data.item_ids[index] for index in item_indices]
    return predictions


def write_predictions(path: str | Path, users: Sequence[str], predictions: dict[str, list[str]]) -> None:
    with open(path, "w", encoding="utf-8", newline="") as stream:
        writer = csv.writer(stream)
        writer.writerow(("user_id", "item_id"))
        for user in users:
            writer.writerow((user, " ".join(predictions[user])))


def _device(name: str):
    if torch is None:
        raise RuntimeError("PyTorch is not installed. Use Kaggle GPU Notebook or install torch.")
    if name == "auto":
        return torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    if name.startswith("cuda") and not torch.cuda.is_available():
        raise RuntimeError("CUDA was requested but is unavailable")
    return torch.device(name)


def _training_loss(model, user_vectors, item_vectors, sampled, regularization: float):
    users, positives, negatives = sampled
    positive_scores = (user_vectors[users] * item_vectors[positives]).sum(dim=1)
    negative_scores = (user_vectors[users] * item_vectors[negatives]).sum(dim=1)
    bpr = F.softplus(negative_scores - positive_scores).mean()
    raw_u = model.user_embedding(users)
    raw_p = model.item_embedding(positives)
    raw_n = model.item_embedding(negatives)
    reg = (raw_u.square().sum(dim=1) + raw_p.square().sum(dim=1) + raw_n.square().sum(dim=1)).mean() / 2
    return bpr + regularization * reg, bpr, reg


def run(args: argparse.Namespace) -> dict:
    device = _device(args.device)
    if args.layers < 1 or args.embedding_dim < 1 or args.epochs < 1:
        raise ValueError("layers, embedding-dim and epochs must be positive")
    if not 0 <= args.edge_dropout < 1 or not 0 <= args.message_dropout < 1:
        raise ValueError("Dropout probabilities must be in [0, 1)")
    if args.reg < 0 or args.lr <= 0:
        raise ValueError("reg must be nonnegative and lr positive")
    torch.manual_seed(args.seed)
    if device.type == "cuda":
        torch.cuda.manual_seed_all(args.seed)
    rng = random.Random(args.seed)
    data = load_data(args.fit, args.users)
    truth = load_truth(args.truth, data.target_users) if args.truth else None
    graph = normalized_graph(data, device)
    if args.model == "lightgcn":
        model = LightGCN(data.num_users, data.num_items, args.embedding_dim, args.layers)
    else:
        model = NGCF(data.num_users, data.num_items, args.embedding_dim, args.layers, args.message_dropout)
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=args.lr)
    print(json.dumps({
        "model": args.model, "device": str(device), "users": data.num_users,
        "items": data.num_items, "fit_edges": data.num_edges,
        "embedding_dim": args.embedding_dim, "layers": args.layers,
        "samples_per_epoch": args.samples_per_epoch,
    }), flush=True)

    best_score = -math.inf
    best_state = None
    best_epoch = None
    stale_evaluations = 0
    last_loss = None
    trained_epochs = 0
    for epoch in range(1, args.epochs + 1):
        trained_epochs = epoch
        start = time.monotonic()
        model.train()
        optimizer.zero_grad(set_to_none=True)
        sampled = sample_triples(data, args.samples_per_epoch, rng, device)
        epoch_graph = sparse_edge_dropout(graph, args.edge_dropout) if args.model == "ngcf" else graph
        user_vectors, item_vectors = model(epoch_graph)
        loss, bpr, reg = _training_loss(model, user_vectors, item_vectors, sampled, args.reg)
        loss.backward()
        optimizer.step()
        last_loss = float(loss.detach().cpu())
        record = {
            "epoch": epoch, "loss": round(last_loss, 6),
            "bpr": round(float(bpr.detach().cpu()), 6),
            "regularization": round(float(reg.detach().cpu()), 6),
            "seconds": round(time.monotonic() - start, 2),
        }
        del sampled, user_vectors, item_vectors, loss, bpr, reg
        should_evaluate = truth is not None and (epoch == 1 or epoch % args.eval_every == 0 or epoch == args.epochs)
        if should_evaluate:
            predictions = predict_top20(model, graph, data, args.eval_user_batch_size)
            metric = recall_at_20(predictions, truth)
            record.update(metric)
            if metric["recall_at_20"] > best_score:
                best_score = metric["recall_at_20"]
                best_state = copy.deepcopy(model.state_dict())
                best_epoch = epoch
                stale_evaluations = 0
                record["best"] = True
            else:
                stale_evaluations += 1
                record["best"] = False
        print(json.dumps(record), flush=True)
        if should_evaluate and args.patience > 0 and stale_evaluations >= args.patience:
            print(json.dumps({"early_stop_epoch": epoch}), flush=True)
            break

    if best_state is not None:
        model.load_state_dict(best_state)
    predictions = predict_top20(model, graph, data, args.eval_user_batch_size)
    write_predictions(args.out, data.target_users, predictions)
    result = {
        "model": args.model, "device": str(device), "fit_edges": data.num_edges,
        "users": len(data.target_users), "output": str(args.out), "last_loss": last_loss,
        "trained_epochs": trained_epochs, "best_epoch": best_epoch,
        "epochs_requested": args.epochs, "seed": args.seed,
        "embedding_dim": args.embedding_dim, "layers": args.layers,
        "samples_per_epoch": args.samples_per_epoch,
        "learning_rate": args.lr, "regularization": args.reg,
        "edge_dropout": args.edge_dropout if args.model == "ngcf" else 0.0,
        "message_dropout": args.message_dropout if args.model == "ngcf" else 0.0,
    }
    if truth is not None:
        result.update(recall_at_20(predictions, truth))
    print(json.dumps({"final": result}), flush=True)
    return result


def main(argv: Sequence[str] | None = None) -> dict:
    parser = argparse.ArgumentParser(description="LightGCN / NGCF on the shared Amazon Top-20 split")
    parser.add_argument("--model", choices=("lightgcn", "ngcf"), required=True)
    parser.add_argument("--fit", required=True, help="val_train.csv or full train.csv")
    parser.add_argument("--users", required=True, help="val_users.csv or test.csv")
    parser.add_argument("--truth", help="val_truth.csv; omit for final test predictions")
    parser.add_argument("--out", required=True, help="Prediction CSV")
    parser.add_argument("--metrics-out", help="Optional JSON file for run settings and results")
    parser.add_argument("--epochs", type=int, default=40)
    parser.add_argument("--embedding-dim", type=int, default=64)
    parser.add_argument("--layers", type=int, default=3)
    parser.add_argument("--samples-per-epoch", type=int, default=100000)
    parser.add_argument("--eval-user-batch-size", type=int, default=256)
    parser.add_argument("--eval-every", type=int, default=5)
    parser.add_argument("--patience", type=int, default=4, help="Validation evaluations without improvement; 0 disables")
    parser.add_argument("--lr", type=float, default=0.001)
    parser.add_argument("--reg", type=float, default=0.0001)
    parser.add_argument("--edge-dropout", type=float, default=0.0, help="Optional NGCF edge dropout")
    parser.add_argument("--message-dropout", type=float, default=0.1, help="NGCF only")
    parser.add_argument("--seed", type=int, default=42)
    parser.add_argument("--device", default="auto", help="auto, cpu, cuda:0")
    args = parser.parse_args(argv)
    if args.eval_every < 1 or args.patience < 0:
        parser.error("eval-every must be positive and patience nonnegative")
    result = run(args)
    if args.metrics_out:
        Path(args.metrics_out).write_text(json.dumps(result, indent=2), encoding="utf-8")
    return result


if __name__ == "__main__":
    main()

## 4. 이전 실험과 동일한 UltraGCN / SimpleX 실행 코드

In [ ]:
%%writefile /kaggle/working/ultra_simplex_runner.py
"""UltraGCN-style and SimpleX-style implicit top-k recommendation.

Designed for the team's provided split and single CUDA GPU (also CPU testable).
Faithfulness notes:
- UltraGCN-style: direct user/item embeddings, degree-aware weighted BCE,
  and item-item co-occurrence constraint. This is a streamlined approximation
  of UltraGCN, not a bit-for-bit reproduction of the published implementation.
- SimpleX-style: user ID + mean item-history tower, cosine contrastive loss,
  leave-one-out history for positive training samples. This is a compact
  implementation of the SimpleX family, not the official codebase.
- Validation positives must NEVER be included in fit history or item neighbors.
"""
from __future__ import annotations

import argparse
import copy
import json
import math
import random
import time
from collections import defaultdict
from pathlib import Path

import numpy as np
import scipy.sparse as sp
import torch
from torch import nn
import torch.nn.functional as F

from graph_models import load_data, load_truth, recall_at_20, write_predictions


def seed_all(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def get_interaction_matrix(data):
    rows = np.asarray(data.positive_users, dtype=np.int64)
    cols = np.asarray(data.positive_items, dtype=np.int64)
    mat = sp.coo_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)),
                        shape=(data.num_users, data.num_items)).tocsr()
    mat.sum_duplicates()
    mat.data[:] = 1.0
    return mat


def ii_neighbors(mat, k: int):
    """Top-k co-occurrence cosine neighbors from FIT ONLY; padded weights=0."""
    n_items = mat.shape[1]
    ids = np.zeros((n_items, k), dtype=np.int64)
    weights = np.zeros((n_items, k), dtype=np.float32)
    deg = np.maximum(np.asarray(mat.getnnz(axis=0), dtype=np.float32), 1)
    co = (mat.T @ mat).tocsr()
    inv = np.sqrt(deg)
    for i in range(n_items):
        start, end = co.indptr[i], co.indptr[i+1]
        nbrs = co.indices[start:end]
        vals = co.data[start:end]
        if not len(nbrs):
            continue
        scores = vals / (inv[i] * inv[nbrs])
        keep = nbrs != i
        nbrs, scores = nbrs[keep], scores[keep]
        if not len(nbrs):
            continue
        take = min(k, len(nbrs))
        if take < len(nbrs):
            pos = np.argpartition(scores, -take)[-take:]
        else:
            pos = np.arange(len(nbrs))
        pos = pos[np.argsort(-scores[pos], kind='stable')]
        ids[i, :take] = nbrs[pos]
        values = scores[pos].astype(np.float32)
        weights[i, :take] = values / max(values.sum(), 1e-9)
    return ids, weights


class UltraStyle(nn.Module):
    def __init__(self, n_users, n_items, dim):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)
        nn.init.normal_(self.user_emb.weight, std=.05)
        nn.init.normal_(self.item_emb.weight, std=.05)

    def vectors(self, mat=None):
        return self.user_emb.weight, self.item_emb.weight


class SimpleXStyle(nn.Module):
    def __init__(self, n_users, n_items, dim, gamma):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)
        self.gamma = gamma
        nn.init.normal_(self.user_emb.weight, std=.05)
        nn.init.normal_(self.item_emb.weight, std=.05)

    def vectors(self, mat):
        # CSR FIT-history aggregation only; do not include held-out labels.
        history = torch.sparse_csr_tensor(
            torch.from_numpy(mat.indptr.astype(np.int64)).to(self.item_emb.weight.device),
            torch.from_numpy(mat.indices.astype(np.int64)).to(self.item_emb.weight.device),
            torch.from_numpy(mat.data.astype(np.float32)).to(self.item_emb.weight.device),
            mat.shape, device=self.item_emb.weight.device)
        lengths = torch.tensor(np.maximum(mat.getnnz(axis=1), 1),
                               dtype=torch.float32, device=self.item_emb.weight.device)
        agg = torch.sparse.mm(history, self.item_emb.weight) / lengths[:, None]
        return (F.normalize(self.user_emb.weight + self.gamma * agg, dim=-1),
                F.normalize(self.item_emb.weight, dim=-1))


def history_sum_for_batch(item_emb, flat_hist, rowptr, users):
    """Differentiable ragged history sum, for selected users only."""
    starts = rowptr[users]
    lengths = rowptr[users + 1] - starts
    offsets = torch.cat([torch.zeros((1,), device=users.device, dtype=torch.long),
                         lengths.cumsum(0)[:-1]])
    owner = torch.repeat_interleave(torch.arange(len(users), device=users.device), lengths)
    flat_positions = torch.repeat_interleave(starts, lengths) + (
        torch.arange(owner.numel(), device=users.device) - offsets[owner]
    )
    bag_idx = flat_hist[flat_positions]
    sums = F.embedding_bag(bag_idx, item_emb.weight, offsets,
                           mode='sum', include_last_offset=False)
    return sums, lengths


def sample_batch(users_all, items_all, n_items, observed_codes, batch, n_negs):
    dev = users_all.device
    sampled = torch.randint(len(users_all), (batch,), device=dev)
    users = users_all[sampled]
    positives = items_all[sampled]
    negatives = torch.randint(n_items, (batch, n_negs), device=dev)
    for _ in range(15):
        ids = users[:, None] * n_items + negatives
        pos = torch.searchsorted(observed_codes, ids.reshape(-1)).clamp_max(len(observed_codes)-1)
        mask = (observed_codes[pos].reshape_as(ids) == ids)
        if not mask.any().item():
            break
        negatives[mask] = torch.randint(n_items, (int(mask.sum().item()),), device=dev)
    else:
        raise RuntimeError('Unable to sample unobserved negatives')
    return users, positives, negatives


def evaluate(model, data, mat, truth, device, batch_size=128):
    model.eval()
    predictions = {}
    with torch.no_grad():
        user_vectors, item_vectors = model.vectors(mat)
        item_trans = item_vectors.T.contiguous()
        for start in range(0, len(data.target_users), batch_size):
            batch_users = data.target_users[start:start+batch_size]
            uidx = [data.user_index[u] for u in batch_users]
            scores = user_vectors[torch.tensor(uidx, device=device)] @ item_trans
            for r, ui in enumerate(uidx):
                seen = data.seen_items.get(ui, set())
                if seen:
                    scores[r, torch.tensor(list(seen), dtype=torch.long,device=device)] = -float('inf')
            top = scores.topk(20, dim=1, sorted=True).indices.cpu().tolist()
            for user, idxs in zip(batch_users, top):
                predictions[user] = [data.item_ids[i] for i in idxs]
        score = recall_at_20(predictions, truth) if truth else None
    return predictions, score


def run(a):
    assert a.epochs > 0 and a.eval_every > 0 and a.batch_size > 0 and a.steps_per_epoch > 0
    assert a.negatives > 0 and a.lr > 0 and a.reg >= 0
    device = torch.device(a.device)
    seed_all(a.seed)
    data = load_data(a.fit, a.users)
    truth = load_truth(a.truth, data.target_users) if a.truth else None
    mat = get_interaction_matrix(data)
    if a.model == 'ultragcn':
        model = UltraStyle(data.num_users, data.num_items, a.dim).to(device)
        user_degree = torch.tensor(np.maximum(mat.getnnz(axis=1), 1), dtype=torch.float32, device=device)
        item_degree = torch.tensor(np.maximum(mat.getnnz(axis=0), 1), dtype=torch.float32, device=device)
        if a.ii_weight > 0:
            ii_id, ii_wt = ii_neighbors(mat, a.ii_topk)
            ii_id = torch.from_numpy(ii_id).to(device)
            ii_wt = torch.from_numpy(ii_wt).to(device)
        else:
            ii_id, ii_wt = None, None
    elif a.model == 'simplex':
        model = SimpleXStyle(data.num_users, data.num_items, a.dim, a.gamma).to(device)
        rowptr = torch.tensor(mat.indptr.astype(np.int64), device=device)
        flat_history = torch.tensor(mat.indices.astype(np.int64), device=device)
    else:
        raise ValueError('Unexpected model')
    optim = torch.optim.Adam(model.parameters(), lr=a.lr)
    u_edges = torch.tensor(data.positive_users, dtype=torch.long, device=device)
    i_edges = torch.tensor(data.positive_items, dtype=torch.long, device=device)
    observed = (u_edges * data.num_items + i_edges).sort().values.unique(sorted=True)

    best_epoch = None
    best_score = -1.0
    best_state = None
    stale = 0
    last_loss = None
    for epoch in range(1, a.epochs+1):
        begin = time.time()
        model.train()
        loss_acc = 0.0
        for _ in range(a.steps_per_epoch):
            u, i, j = sample_batch(u_edges, i_edges, data.num_items, observed,
                                   a.batch_size, a.negatives)
            optim.zero_grad(set_to_none=True)
            if a.model == 'ultragcn':
                uv = model.user_emb(u)
                pv = model.item_emb(i)
                nv = model.item_emb(j)
                pos = (uv * pv).sum(-1)
                neg = (uv[:,None,:] * nv).sum(-1)
                # Degree-aware confidence for user-item constraint.
                confidence = (user_degree[u] / item_degree[i]).sqrt().clamp(0.25, 4.0)
                bce_pos = (1.0 + a.beta * confidence) * F.softplus(-pos)
                bce_neg = F.softplus(neg).mean(-1) * a.neg_weight
                loss = (bce_pos + bce_neg).mean()
                if ii_id is not None:
                    nbr_emb = model.item_emb(ii_id[i])
                    nbr_s = (uv[:,None,:] * nbr_emb).sum(-1)
                    item_loss = (F.softplus(-nbr_s) * ii_wt[i]).sum(-1).mean()
                    loss = loss + a.ii_weight * item_loss
                reg = .5*(uv.square().sum(-1).mean()+pv.square().sum(-1).mean()+nv.square().sum(-1).mean())
            else:
                # Leave the current target item OUT of the user's training history.
                history_sum, lengths = history_sum_for_batch(model.item_emb, flat_history, rowptr, u)
                agg_loo = (history_sum - model.item_emb(i)) / (lengths-1).clamp_min(1)[:,None]
                user_r = F.normalize(model.user_emb(u) + model.gamma * agg_loo, dim=-1)
                p = F.normalize(model.item_emb(i), dim=-1)
                n = F.normalize(model.item_emb(j), dim=-1)
                pos_s = (user_r * p).sum(-1)
                neg_s = (user_r[:,None,:] * n).sum(-1)
                loss = ((1.0 - pos_s) + a.neg_weight * F.relu(neg_s-a.margin).square().mean(-1)).mean()
                reg = .5*(model.user_emb(u).square().sum(-1).mean()+model.item_emb(i).square().sum(-1).mean()+model.item_emb(j).square().sum(-1).mean())
            loss = loss + a.reg * reg
            if not torch.isfinite(loss).item():
                raise RuntimeError('Nonfinite loss')
            loss.backward()
            optim.step()
            loss_acc += loss.item()
        last_loss = loss_acc / a.steps_per_epoch
        if (epoch % a.eval_every == 0 or epoch == a.epochs or epoch == 1) and truth:
            preds, result = evaluate(model, data, mat, truth, device, a.eval_batch)
            best = result['recall_at_20'] > best_score
            if best:
                best_score = result['recall_at_20']
                best_epoch = epoch
                best_state = copy.deepcopy({k:v.detach().cpu() for k,v in model.state_dict().items()})
                stale = 0
            else:
                stale += 1
            print(json.dumps({'epoch':epoch,'loss':last_loss,'recall_at_20':result['recall_at_20'],
                              'hits':result['hits'],'best':best,'seconds':round(time.time()-begin,2)}),flush=True)
            if a.patience>0 and stale>=a.patience:
                break
        elif epoch % max(a.eval_every, 10) == 0 or epoch == a.epochs:
            print(json.dumps({'epoch':epoch,'loss':last_loss,'seconds':round(time.time()-begin,2)}),flush=True)

    if best_state is not None:
        model.load_state_dict(best_state)
    predictions, result = evaluate(model, data, mat, truth, device, a.eval_batch)
    write_predictions(a.out, data.target_users, predictions)
    output = dict(model=a.model, out=a.out, fit=a.fit, best_epoch=best_epoch,
                  trained_epochs=epoch, epochs_requested=a.epochs, last_loss=last_loss,
                  dim=a.dim, lr=a.lr, negatives=a.negatives, reg=a.reg,
                  steps_per_epoch=a.steps_per_epoch, batch_size=a.batch_size,
                  ii_weight=a.ii_weight, beta=a.beta, gamma=a.gamma,
                  margin=a.margin, neg_weight=a.neg_weight, seed=a.seed)
    if result:
        output.update(result)
    if a.metrics:
        Path(a.metrics).write_text(json.dumps(output,indent=2),encoding='utf-8')
    print(json.dumps({'final':output}), flush=True)
    return output


def main():
    p=argparse.ArgumentParser(description='UltraGCN-style and SimpleX-style collaborative filtering')
    p.add_argument('--model', choices=['ultragcn','simplex'], required=True)
    p.add_argument('--fit', required=True)
    p.add_argument('--users', required=True)
    p.add_argument('--truth', default=None)
    p.add_argument('--out', required=True)
    p.add_argument('--metrics', default=None)
    p.add_argument('--epochs',type=int,default=70)
    p.add_argument('--eval-every',type=int,default=10)
    p.add_argument('--patience',type=int,default=5)
    p.add_argument('--device',default='cuda:0')
    p.add_argument('--dim',type=int,default=128)
    p.add_argument('--batch-size',type=int,default=2048)
    p.add_argument('--steps-per-epoch',type=int,default=40)
    p.add_argument('--negatives',type=int,default=10)
    p.add_argument('--lr',type=float,default=.002)
    p.add_argument('--reg',type=float,default=1e-5)
    p.add_argument('--ii-weight',type=float,default=.1)
    p.add_argument('--ii-topk',type=int,default=10)
    p.add_argument('--beta',type=float,default=.1)
    p.add_argument('--neg-weight',type=float,default=4.)
    p.add_argument('--gamma',type=float,default=1.)
    p.add_argument('--margin',type=float,default=.2)
    p.add_argument('--seed',type=int,default=42)
    p.add_argument('--eval-batch',type=int,default=128)
    a=p.parse_args()
    run(a)


if __name__=='__main__':
    main()


## 5. 최종 모델 설정 및 Full-Train 재학습

이전 **동일한 Validation**에서 찾은 최고 Epoch만 사용합니다. Test 정답이 없으므로 Full-Train 모델은 Early Stopping으로 Epoch를 재선택하지 않습니다.

같은 Kaggle 런타임에서 재실행하면 유효한 파일을 재사용합니다. 새 런타임에서는 재학습이 필요할 수 있습니다. `top3_final/*.log`에 전체 학습 로그가 남습니다. **셀 실행 중에는 수동으로 중단하지 마세요.**

In [ ]:
from teamkit import read_users, read_interactions, read_predictions, validate_predictions

# 정확히 실험에서 선택됐던 3개 모델의 설정만 재현
SPECS = {
 'lightgcn': {
    'model':'lightgcn', 'runner':'graph', 'epochs':575,
    'dim':128, 'layers':3, 'lr':.003, 'reg':.0001,
    'samples_per_epoch':100000,
 },
 'simplex': {
    'model':'simplex', 'runner':'new', 'epochs':650,
    'dim':128, 'lr':.002, 'reg':.00001, 'negatives':20,
    'gamma':.5, 'margin':.20, 'neg_weight':4.,
    'steps_per_epoch':35, 'batch_size':2048,
 },
 'ultragcn': {
    'model':'ultragcn', 'runner':'new', 'epochs':80,
    'dim':128, 'lr':.001, 'reg':.00001, 'negatives':10,
    'ii_weight':.30, 'beta':.1, 'neg_weight':4.,
    'steps_per_epoch':35, 'batch_size':2048,
 },
}
FULL_USER_IDS=read_users(TEST)
FULL_SEEN,FULL_POP=read_interactions(TRAIN)
FULL_CATALOG=set(FULL_POP)
assert len(FULL_USER_IDS)>0 and len(FULL_CATALOG)>20
print('Test 사용자 수:',len(FULL_USER_IDS))
print('학습 데이터 아이템 수:',len(FULL_CATALOG))
print('학습 설정:')
for name,spec in SPECS.items(): print(name, 'Epoch',spec['epochs'],spec)


def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as stream:
        for chunk in iter(lambda:stream.read(2*1024*1024),b''):h.update(chunk)
    return h.hexdigest()

SOURCE_SHAS = {name:sha256_file(WORK/name) for name in ('teamkit.py','graph_models.py','ultra_simplex_runner.py')}
FINGERPRINT_SOURCE = {
 'train_sha256':sha256_file(TRAIN), 'test_sha256':sha256_file(TEST),
 'source_sha256':SOURCE_SHAS, 'seed':SEED,
}


def create_command(spec,out,metrics):
    common=['--fit',str(TRAIN),'--users',str(TEST),
            '--out',str(out),'--epochs',str(spec['epochs']),
            '--lr',str(spec['lr']),'--reg',str(spec['reg']),
            '--seed',str(SEED),'--device',DEVICE]
    if spec['runner']=='graph':
        return [sys.executable,str(WORK/'graph_models.py'),'--model',spec['model'],
                *common,'--metrics-out',str(metrics),
                '--embedding-dim',str(spec['dim']),'--layers',str(spec['layers']),
                '--samples-per-epoch',str(spec['samples_per_epoch']),
                '--eval-every','25','--patience','0']
    return [sys.executable,str(WORK/'ultra_simplex_runner.py'),
            '--model',spec['model'],*common,'--metrics',str(metrics),
            '--dim',str(spec['dim']),'--batch-size',str(spec['batch_size']),
            '--steps-per-epoch',str(spec['steps_per_epoch']),
            '--negatives',str(spec['negatives']),
            '--ii-weight',str(spec.get('ii_weight',0)),
            '--beta',str(spec.get('beta',.1)),
            '--neg-weight',str(spec.get('neg_weight',4.)),
            '--gamma',str(spec.get('gamma',1.)),
            '--margin',str(spec.get('margin',.2)),
            '--eval-every','25','--patience','0']


def run_full_model(name):
    spec=SPECS[name]
    pred=OUT/f'{name}_full.csv'
    metrics=OUT/f'{name}_full.metrics.json'
    stamp=OUT/f'{name}_full.config.json'
    log=OUT/f'{name}_full.log'
    expected={**FINGERPRINT_SOURCE,'spec':spec}
    if pred.is_file() and metrics.is_file() and stamp.is_file():
        try:
            if json.loads(stamp.read_text(encoding='utf-8')) == expected:
                pp=read_predictions(pred)
                validate_predictions(pp,FULL_USER_IDS,FULL_SEEN,FULL_CATALOG)
                print('✅ 유효한 Full-Train 캐시 재사용:',name,'|',pred)
                return pred
        except Exception as exc:
            print('캐시 무효, 재학습:',name,repr(exc)[:200])
    # 이전 불완전 파일을 같은 설정으로 오인하지 않도록 미리 삭제
    pred.unlink(missing_ok=True)
    metrics.unlink(missing_ok=True)
    stamp.unlink(missing_ok=True)
    cmd=create_command(spec,pred,metrics)
    print(f'🚀 Full-Train 시작: {name} / {spec["epochs"]} Epoch',flush=True)
    with log.open('w',encoding='utf-8') as fh:
        proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                              text=True,bufsize=1)
        for line in proc.stdout:
            fh.write(line)
            if ('"epoch"' in line and ('"recall_at_20"' in line or
                                        ('"epoch": '+str(spec['epochs'])+',') in line)):
                print(line.rstrip()[:500],flush=True)
            if 'Traceback' in line or 'Error' in line or 'OOM' in line:
                print(line.rstrip()[:500],flush=True)
        returncode=proc.wait()
    if returncode != 0 or not pred.is_file() or not metrics.is_file():
        raise RuntimeError(f'{name} Full-Train 실패; 로그 확인: {log} / exit={returncode}')
    predictions=read_predictions(pred)
    validate_predictions(predictions,FULL_USER_IDS,FULL_SEEN,FULL_CATALOG)
    metric_data=json.loads(metrics.read_text(encoding='utf-8'))
    if int(metric_data.get('trained_epochs',-1)) != int(spec['epochs']):
        raise RuntimeError(f'{name} 학습 Epoch 불일치: {metric_data}')
    stamp.write_text(json.dumps(expected,ensure_ascii=False,indent=2),encoding='utf-8')
    print('✅ 완료:',name,'|',pred,'| Epoch:',spec['epochs'],flush=True)
    return pred

## 6. 세 모델을 각각 딱 한 번씩 전체 train에 학습

In [ ]:
FULL_PREDICTION_FILES={}
for name in ('lightgcn','simplex','ultragcn'):
    FULL_PREDICTION_FILES[name]=run_full_model(name)
print('✅ 3개 기반 모델 Full-Train 예측 준비 완료!')

## 7. Validation 1~3위 앙상블 생성 · 제출 형식 3회 검증

검증 실험의 `fuse_user` 계산 방식을 정확히 재사용합니다. 단, 학습·예측은 **전체 train.csv 기반**으로 새로 만든 결과를 사용합니다.

**경고:** 1위와 2위의 `k` 차이는 후보 순위를 조금 바꿀 뿐이므로 Test 성능이 크게 다르지 않을 수 있습니다.

In [ ]:
from teamkit import write_predictions
PRED={name:read_predictions(path) for name,path in FULL_PREDICTION_FILES.items()}
assert all(set(p)==set(FULL_USER_IDS) for p in PRED.values())

SUBMISSION_SPECS=[
 dict(rank=1,name='E_L_X_U_k60_b',members=['lightgcn','simplex','ultragcn'],
      weights=[.35,.50,.15],method='rrf',k=60,
      validation_recall_at_20=.042812),
 dict(rank=2,name='E_L_X_U_k20_b',members=['lightgcn','simplex','ultragcn'],
      weights=[.35,.50,.15],method='rrf',k=20,
      validation_recall_at_20=.042768),
 dict(rank=3,name='E_L_simplex_top1_k20_L50',members=['lightgcn','simplex'],
      weights=[.50,.50],method='rrf',k=20,
      validation_recall_at_20=.041601),
]


def iid_key(item):
    return (0,int(item)) if item.isdecimal() else (1,item)


def fuse_one(user,cfg):
    scores=defaultdict(float)
    for name,w in zip(cfg['members'],cfg['weights']):
        for rank,item in enumerate(PRED[name][user],1):
            scores[item]+=float(w)/(int(cfg['k'])+rank)
    return sorted(scores,key=lambda item:(-scores[item],iid_key(item)))[:20]


def check_final_file(filepath):
    # 팀 키트 공식 제출 검사. False나 에러로 넘어가지 않음
    cmd=[sys.executable,str(WORK/'teamkit.py'),'check',
         '--pred',str(filepath),'--train',str(TRAIN),'--test',str(TEST)]
    result=subprocess.run(cmd,capture_output=True,text=True)
    if result.returncode:
        print(result.stdout[-1500:]);print(result.stderr[-1500:])
        raise RuntimeError(f'제출 형식 검사 실패: {filepath}')
    print('  check:',result.stdout.strip().replace('\n',' '))


SUBMISSION_ROWS=[]
for cfg in SUBMISSION_SPECS:
    dest=WORK/f'submission_top{cfg["rank"]}.csv'
    print('만드는 중:',dest,'|',cfg['name'],flush=True)
    predictions={u:fuse_one(u,cfg) for u in FULL_USER_IDS}
    validate_predictions(predictions,FULL_USER_IDS,FULL_SEEN,FULL_CATALOG)
    write_predictions(dest,FULL_USER_IDS,predictions)
    check_final_file(dest)
    SUBMISSION_ROWS.append(dict(rank=cfg['rank'],file=str(dest),
                                name=cfg['name'],
                                validation_recall_at_20=cfg['validation_recall_at_20'],
                                size_bytes=dest.stat().st_size))
    print('✅ 제출 CSV 준비:',dest.name,'| 사용자 수:',len(predictions),flush=True)
    del predictions

META=OUT/'top3_submission_configs.json'
META.write_text(json.dumps({'sources':FINGERPRINT_SOURCE,
  'model_specs':SPECS,'ensemble_specs':SUBMISSION_SPECS,
  'files':SUBMISSION_ROWS},ensure_ascii=False,indent=2),encoding='utf-8')

BUNDLE=WORK/'top3_submissions.zip'
with zipfile.ZipFile(BUNDLE,'w',zipfile.ZIP_DEFLATED) as z:
    for row in SUBMISSION_ROWS:
        f=Path(row['file'])
        z.write(f,arcname=f.name)
    z.write(META,arcname=META.name)
print('\n🎉 TOP 1 / TOP 2 / TOP 3 파일 생성 + 검사 완료!')
print('한 번에 다운로드할 ZIP:',BUNDLE)
display(pd.DataFrame(SUBMISSION_ROWS))
print('ZIP 크기 MB:',round(BUNDLE.stat().st_size/1024**2,2))

## 8. 다운로드 및 Kaggle 제출

**가장 편한 방법:** Kaggle 노트북 오른쪽 **Output → `/kaggle/working/top3_submissions.zip`**을 다운로드하고 압축을 풉니다. ZIP에는 `submission_top1.csv`, `submission_top2.csv`, `submission_top3.csv`가 모두 들어 있습니다.

파일을 세 번 **각각 별도로** Kaggle 대회 → `Submit Predictions`에서 업로드하세요. 반드시 CSV 자체를 제출해야 하며 **ZIP 파일을 업로드하면 안 됩니다.**

- Top 1 설명: `L+X+U RRF k60 35-50-15 (full train)`
- Top 2 설명: `L+X+U RRF k20 35-50-15 (full train)`
- Top 3 설명: `L+X RRF k20 50-50 (full train)`

Kaggle에서 `FileLink`가 404인 경우가 있으므로 다운로드는 **Output 파일 목록**을 우선 사용하세요. Output에 안 보이면 세션에서 파일 생성과 `Save Version` 상태를 확인해야 합니다. `Save & Run All`은 긴 학습을 다시 시작할 수 있습니다.

**공식 데이터에 외부 정보는 추가하지 않습니다.** Test 정답은 사용하지 않습니다.